# Top-K Token Analysis for Dual-Stream Model

This notebook analyzes the top-K predicted tokens during generation to investigate the hypothesis that the model generates "average" or "compromise" tokens that don't fit either context well.

## Analysis Goals:
1. Examine which tokens appear in top-K predictions
2. Measure cosine similarity between embeddings of top-K tokens
3. Investigate whether high-probability tokens have embeddings that are "in between" the two streams
4. Understand why generated text appears incoherent despite low loss


In [1]:
import sys
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer
from pathlib import Path

# Add project root to path
project_root = Path('/disk/4tb/tikhonov/projects/mixture-of-embeddings')
sys.path.insert(0, str(project_root))

from model import DualStreamLLM

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully!")


Libraries imported successfully!


## 1. Load Model and Checkpoint


In [ ]:
# Configuration
checkpoint_path = '/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251228_230459/checkpoint-160000/model.pt'
# checkpoint_path = '/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251224_185822/checkpoint-200000/model.pt'
model_name = 'EleutherAI/pythia-2.8b'
objective = 'separate'

shared_embeddings = True
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

print(f"Device: {device}")
print(f"Model: {model_name}")
print(f"Objective: {objective}")
print(f"Shared embeddings: {shared_embeddings}")


Device: cuda:1
Model: EleutherAI/pythia-2.8b
Objective: separate
Shared embeddings: True


In [3]:
# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocab size: {len(tokenizer)}")
print(f"EOS token: {tokenizer.eos_token} (id: {tokenizer.eos_token_id})")
print(f"PAD token: {tokenizer.pad_token} (id: {tokenizer.pad_token_id})")


Loading tokenizer...
Vocab size: 50277
EOS token: <|endoftext|> (id: 0)
PAD token: <|endoftext|> (id: 0)


In [5]:
# Load model
print("Loading model...")
model = DualStreamLLM(
    model_name=model_name,
    freeze_backbone=False,
    embedding_init_strategy='copy',
    embedding_init_noise_std=0.01,
    objective=objective,
    shared_embeddings=shared_embeddings
)

# Load checkpoint
print(f"Loading checkpoint from {checkpoint_path}...")
checkpoint = torch.load(checkpoint_path, map_location='cpu')
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print("Model loaded successfully!")
print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters")


Loading model...
Loading checkpoint from /disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251228_230459/checkpoint-160000/model.pt...


/tmp/ipykernel_2307232/2847563851.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location='cpu')


Model loaded successfully!
Model has 3,174,650,880 parameters


## 2. Define Test Prompts and Helper Functions


In [89]:
# Test prompts (same as validation prompts)
test_prompts = [
    # ("The capital of", "Machine learning is"),
    # ("Once upon a time", "In the field of"),
    # ("The history of", "Modern technology enables"),
    
    
    
    ("apple banana raspberry cherry", "physics chemistry mathematics biology")
    
    # ("The orchestra performed Beethoven's Symphony No 9 with great passion,", 
    #  "The new quantum computer processed a lot of the complex calculations in milliseconds.",),
    
    # ("Possible Categories: Music, Sports, Technology, Travel.\n\nText: The orchestra performed Beethoven's Symphony No 9 with great passion.\nCategory:", 
    #  "Possible Categories: Music, Sports, Technology, Travel.\n\nText: The new quantum computer processed a lot of the complex calculations in milliseconds.\nCategory:",)
]

print("Test prompts:")
for i, (prompt_a, prompt_b) in enumerate(test_prompts, 1):
    print(f"{i}. A: '{prompt_a}' | B: '{prompt_b}'")
    print(len(tokenizer.tokenize(prompt_a)), len(tokenizer.tokenize(prompt_b)))


Test prompts:
1. A: 'apple banana raspberry cherry' | B: 'physics chemistry mathematics biology'
5 5


In [91]:
# Generate samples from the model
from generation import generate_samples

print("Generating samples...")
print("=" * 80)

samples = generate_samples(
    model, tokenizer, test_prompts, 
    device, max_new_tokens=50, 
    temperature=0.1, 
    objective=objective
)

for i, sample in enumerate(samples, 1):
    print(f"\n[Sample {i}:]")
    print(f"  [Prompt A:] {sample['prompt_A']}")
    print(f"  [Generated A:] {sample['generated_A'][len(sample['prompt_A']):]}")
    print(f"  [Prompt B:] {sample['prompt_B']}")
    print(f"  [Generated B:] {sample['generated_B'][len(sample['prompt_B']):]}")

print("\n" + "=" * 80)


Generating samples...

[Sample 1:]
  [Prompt A:] apple banana raspberry cherry
  [Generated A:] 


















































  [Prompt B:] physics chemistry mathematics biology
  [Generated B:] , and the science of the science of the science of the science of the science of the science of the science of the science of the science of the science of the science of the science of the science of the science of the science of the science of



In [92]:
def get_topk_tokens_analysis(model, tokenizer, prompt_a, prompt_b, k=10, max_steps=5, temperature=0.0):
    """
    Analyze top-K tokens at each generation step.
    
    Returns:
        List of dicts, one per generation step, containing:
        - step: generation step number
        - topk_tokens: list of token strings
        - topk_ids: list of token ids
        - topk_probs: list of probabilities
        - topk_logits_a: logits from head A
        - topk_logits_b: logits from head B
    """
    model.eval()
    
    # Tokenize prompts
    encoded_a = tokenizer(prompt_a, return_tensors='pt').to(device)
    encoded_b = tokenizer(prompt_b, return_tensors='pt').to(device)
    
    input_ids_a = encoded_a['input_ids']
    attention_mask_a = encoded_a['attention_mask']
    input_ids_b = encoded_b['input_ids']
    attention_mask_b = encoded_b['attention_mask']
    
    analysis_results = []
    
    with torch.no_grad():
        for step in range(max_steps):
            # Forward pass
            logits_a, logits_b = model(
                input_ids_A=input_ids_a,
                attention_mask_A=attention_mask_a,
                input_ids_B=input_ids_b,
                attention_mask_B=attention_mask_b
            )
            
            # Get logits for next token (last position)
            next_logits_a = logits_a[:, -1, :] / (temperature + 1e-4)
            next_logits_b = logits_b[:, -1, :] / (temperature + 1e-4)
            
            # Get probabilities
            probs_a = F.softmax(next_logits_a, dim=-1)
            probs_b = F.softmax(next_logits_b, dim=-1)
            
            # Get top-K for head A
            topk_probs_a, topk_ids_a = torch.topk(probs_a, k, dim=-1)
            topk_logits_a = next_logits_a[0, topk_ids_a[0]]
            
            # Get top-K for head B
            topk_probs_b, topk_ids_b = torch.topk(probs_b, k, dim=-1)
            topk_logits_b = next_logits_b[0, topk_ids_b[0]]
            
            # Decode tokens
            topk_tokens_a = [tokenizer.decode([tid]) for tid in topk_ids_a[0]]
            topk_tokens_b = [tokenizer.decode([tid]) for tid in topk_ids_b[0]]
            
            # Store analysis for this step
            analysis_results.append({
                'step': step,
                'head': 'A',
                'prompt': prompt_a,
                'current_text': tokenizer.decode(input_ids_a[0]),
                'topk_tokens': topk_tokens_a,
                'topk_ids': topk_ids_a[0].cpu().numpy(),
                'topk_probs': topk_probs_a[0].float().cpu().numpy(),
                'topk_logits': topk_logits_a.float().cpu().numpy(),
            })
            
            analysis_results.append({
                'step': step,
                'head': 'B',
                'prompt': prompt_b,
                'current_text': tokenizer.decode(input_ids_b[0]),
                'topk_tokens': topk_tokens_b,
                'topk_ids': topk_ids_b[0].cpu().numpy(),
                'topk_probs': topk_probs_b[0].float().cpu().numpy(),
                'topk_logits': topk_logits_b.float().cpu().numpy(),
            })
            
            # Sample next token (using head A for continuation)
            next_token_a = torch.multinomial(probs_a, num_samples=1)
            next_token_b = torch.multinomial(probs_b, num_samples=1)
            
            # Append to sequences
            input_ids_a = torch.cat([input_ids_a, next_token_a], dim=1)
            input_ids_b = torch.cat([input_ids_b, next_token_b], dim=1)
            attention_mask_a = torch.cat([attention_mask_a, torch.ones((1, 1), device=device)], dim=1)
            attention_mask_b = torch.cat([attention_mask_b, torch.ones((1, 1), device=device)], dim=1)
    
    return analysis_results

print("Helper functions defined!")


Helper functions defined!


## 3. Run Analysis on Test Prompts


In [93]:
# Run analysis on first prompt pair
prompt_a, prompt_b = test_prompts[0]
print(f"Analyzing: A='{prompt_a}' | B='{prompt_b}'")
print("=" * 80)


k = 10  # Number of top tokens to analyze
max_steps = 3  # Number of generation steps

results = get_topk_tokens_analysis(model, tokenizer, prompt_a, prompt_b, k=k, max_steps=max_steps, temperature=0.7)

print(f"\nCollected {len(results)} analysis entries ({max_steps} steps × 2 heads)")


Analyzing: A='apple banana raspberry cherry' | B='physics chemistry mathematics biology'

Collected 6 analysis entries (3 steps × 2 heads)


## 4. Visualize Top-K Tokens


In [94]:
checkpoint_path

'/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251228_230459/checkpoint-160000/model.pt'

In [95]:
# Display top-K tokens for each step and head
for result in results:
    print(f"\n{'='*80}")
    print(f"Step {result['step']} | Head {result['head']} | Prompt: '{result['prompt']}'")
    print(f"Current text: {result['current_text']}")
    print(f"{'='*80}")
    
    # Create a DataFrame for nice display
    df = pd.DataFrame({
        'Rank': range(1, len(result['topk_tokens']) + 1),
        'Token': result['topk_tokens'],
        'Token ID': result['topk_ids'],
        'Probability': result['topk_probs'],
        'Log Prob': np.log(result['topk_probs']),
    })
    
    # Format probabilities
    df['Probability'] = df['Probability'].apply(lambda x: f"{x:.4f}")
    df['Log Prob'] = df['Log Prob'].apply(lambda x: f"{x:.4f}")
    
    print(df.to_string(index=False))
    print()



Step 0 | Head A | Prompt: 'apple banana raspberry cherry'
Current text: apple banana raspberry cherry
 Rank      Token  Token ID Probability Log Prob
    1     banana     36767      0.0928  -2.3776
    2         \n       187      0.0928  -2.3776
    3      apple     19126      0.0820  -2.5007
    4        and       285      0.0635  -2.7571
    5          ,        13      0.0496  -3.0046
    6      fruit      9279      0.0234  -3.7534
    7  chocolate     14354      0.0234  -3.7534
    8          .        15      0.0234  -3.7534
    9        ice      6749      0.0183  -4.0003
   10     orange     13735      0.0183  -4.0003


Step 0 | Head B | Prompt: 'physics chemistry mathematics biology'
Current text: physics chemistry mathematics biology
 Rank     Token  Token ID Probability Log Prob
    1         ,        13      0.1562  -1.8563
    2  research      2561      0.1562  -1.8563
    3       and       285      0.1562  -1.8563
    4        \n       187      0.1216  -2.1072
    5         

## 5. Embedding Similarity Analysis

Now let's analyze the embeddings of the top-K tokens to see if they are semantically similar or if high-probability tokens have "average" embeddings.


In [14]:
def compute_embedding_similarity_matrix(token_ids, embedding_layer):
    """
    Compute pairwise cosine similarity between embeddings of given token IDs.
    
    Args:
        token_ids: array of token IDs
        embedding_layer: nn.Embedding layer
        
    Returns:
        similarity_matrix: (len(token_ids), len(token_ids)) cosine similarity matrix
    """
    # Get embeddings
    with torch.no_grad():
        token_ids_tensor = torch.tensor(token_ids, device=device)
        embeddings = embedding_layer(token_ids_tensor)  # (num_tokens, hidden_dim)
        
        # Compute cosine similarity
        embeddings_norm = F.normalize(embeddings, p=2, dim=1)
        similarity_matrix = torch.mm(embeddings_norm, embeddings_norm.t())
        
    return similarity_matrix.cpu().numpy()


def plot_similarity_heatmap(similarity_matrix, tokens, title="Token Embedding Similarity"):
    """Plot heatmap of similarity matrix."""
    plt.figure(figsize=(10, 8))
    
    # Create labeled version for better readability
    token_labels = [f"{i+1}:{t}" for i, t in enumerate(tokens)]
    
    sns.heatmap(
        similarity_matrix,
        xticklabels=token_labels,
        yticklabels=token_labels,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        vmin=0,
        vmax=1,
        center=0.5,
        square=True,
        cbar_kws={'label': 'Cosine Similarity'}
    )
    
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

print("Embedding similarity functions defined!")


Embedding similarity functions defined!


In [15]:
# Analyze embeddings for each step
# Use the shared embedding layer (since shared_embeddings=True)
embedding_layer = model.emb

for result in results[:4]:  # Show first 2 steps (2 steps × 2 heads = 4 entries)
    step = result['step']
    head = result['head']
    tokens = result['topk_tokens']
    token_ids = result['topk_ids']
    probs = result['topk_probs']
    
    print(f"\n{'='*80}")
    print(f"Embedding Analysis: Step {step} | Head {head}")
    print(f"Prompt: '{result['prompt']}'")
    print(f"{'='*80}\n")
    
    # Compute similarity matrix
    sim_matrix = compute_embedding_similarity_matrix(token_ids, embedding_layer)
    
    # Print statistics
    # Exclude diagonal (self-similarity = 1.0)
    off_diagonal_sims = sim_matrix[~np.eye(sim_matrix.shape[0], dtype=bool)]
    
    print(f"Similarity statistics (excluding diagonal):")
    print(f"  Mean: {off_diagonal_sims.mean():.4f}")
    print(f"  Std:  {off_diagonal_sims.std():.4f}")
    print(f"  Min:  {off_diagonal_sims.min():.4f}")
    print(f"  Max:  {off_diagonal_sims.max():.4f}")
    
    # Find most similar pairs (excluding self)
    print(f"\nMost similar token pairs:")
    for i in range(len(tokens)):
        for j in range(i+1, len(tokens)):
            if sim_matrix[i, j] > 0.8:  # High similarity threshold
                print(f"  '{tokens[i]}' ↔ '{tokens[j]}': {sim_matrix[i, j]:.4f} (probs: {probs[i]:.4f}, {probs[j]:.4f})")
    
    # Plot heatmap
    plot_similarity_heatmap(
        sim_matrix,
        tokens,
        title=f"Token Embedding Similarity - Step {step}, Head {head}"
    )



Embedding Analysis: Step 0 | Head A
Prompt: 'The capital of'



TypeError: Got unsupported ScalarType BFloat16

### 5.1 Cross-Stream Analysis: Are Top Tokens "In Between"?

Let's compare the top tokens from both heads to see if there's overlap or if they're selecting similar tokens (which might indicate "compromise" tokens).


In [ ]:
# Compare top tokens between heads A and B at each step
for step in range(max_steps):
    # Get results for this step
    result_a = [r for r in results if r['step'] == step and r['head'] == 'A'][0]
    result_b = [r for r in results if r['step'] == step and r['head'] == 'B'][0]
    
    tokens_a = set(result_a['topk_tokens'])
    tokens_b = set(result_b['topk_tokens'])
    
    # Find overlapping tokens
    overlap = tokens_a & tokens_b
    
    print(f"\n{'='*80}")
    print(f"Step {step} - Token Overlap Analysis")
    print(f"{'='*80}")
    print(f"Head A prompt: '{result_a['prompt']}'")
    print(f"Head B prompt: '{result_b['prompt']}'")
    print(f"\nTop-{k} token overlap: {len(overlap)}/{k} tokens")
    
    if overlap:
        print(f"Overlapping tokens: {overlap}")
        
        # For overlapping tokens, compare their ranks and probabilities
        print(f"\nOverlapping token details:")
        for token in overlap:
            idx_a = result_a['topk_tokens'].index(token)
            idx_b = result_b['topk_tokens'].index(token)
            prob_a = result_a['topk_probs'][idx_a]
            prob_b = result_b['topk_probs'][idx_b]
            
            print(f"  '{token}':")
            print(f"    Head A: rank {idx_a+1}, prob {prob_a:.4f}")
            print(f"    Head B: rank {idx_b+1}, prob {prob_b:.4f}")
            print(f"    Prob ratio (A/B): {prob_a/prob_b:.4f}")
    else:
        print("No overlapping tokens in top-K!")
    
    print()


### 5.2 Cross-Head Embedding Similarity

Now let's compute embedding similarities between top tokens from Head A and Head B to see if they're semantically close (which would support the "compromise token" hypothesis).


In [ ]:
def compute_cross_head_similarity(token_ids_a, token_ids_b, embedding_layer):
    """
    Compute cosine similarity between embeddings from two sets of token IDs.
    
    Returns:
        cross_similarity: (len(token_ids_a), len(token_ids_b)) matrix
    """
    with torch.no_grad():
        ids_a = torch.tensor(token_ids_a, device=device)
        ids_b = torch.tensor(token_ids_b, device=device)
        
        emb_a = embedding_layer(ids_a)
        emb_b = embedding_layer(ids_b)
        
        # Normalize
        emb_a_norm = F.normalize(emb_a, p=2, dim=1)
        emb_b_norm = F.normalize(emb_b, p=2, dim=1)
        
        # Cross similarity
        cross_sim = torch.mm(emb_a_norm, emb_b_norm.t())
        
    return cross_sim.cpu().numpy()


# Analyze cross-head similarities for first 2 steps
for step in range(min(2, max_steps)):
    result_a = [r for r in results if r['step'] == step and r['head'] == 'A'][0]
    result_b = [r for r in results if r['step'] == step and r['head'] == 'B'][0]
    
    print(f"\n{'='*80}")
    print(f"Cross-Head Embedding Similarity - Step {step}")
    print(f"{'='*80}")
    print(f"Head A: '{result_a['prompt']}'")
    print(f"Head B: '{result_b['prompt']}'")
    print()
    
    # Compute cross-head similarity
    cross_sim = compute_cross_head_similarity(
        result_a['topk_ids'],
        result_b['topk_ids'],
        embedding_layer
    )
    
    # Statistics
    print(f"Cross-head embedding similarity statistics:")
    print(f"  Mean: {cross_sim.mean():.4f}")
    print(f"  Std:  {cross_sim.std():.4f}")
    print(f"  Min:  {cross_sim.min():.4f}")
    print(f"  Max:  {cross_sim.max():.4f}")
    
    # Find highly similar cross-head pairs
    print(f"\nHighly similar token pairs between heads (similarity > 0.7):")
    for i, token_a in enumerate(result_a['topk_tokens']):
        for j, token_b in enumerate(result_b['topk_tokens']):
            if cross_sim[i, j] > 0.7:
                print(f"  A[{i+1}]:'{token_a}' ↔ B[{j+1}]:'{token_b}': {cross_sim[i, j]:.4f}")
                print(f"    Probs: A={result_a['topk_probs'][i]:.4f}, B={result_b['topk_probs'][j]:.4f}")
    
    # Plot cross-head heatmap
    plt.figure(figsize=(12, 10))
    
    token_labels_a = [f"A{i+1}:{t}" for i, t in enumerate(result_a['topk_tokens'])]
    token_labels_b = [f"B{i+1}:{t}" for i, t in enumerate(result_b['topk_tokens'])]
    
    sns.heatmap(
        cross_sim,
        xticklabels=token_labels_b,
        yticklabels=token_labels_a,
        annot=True,
        fmt='.2f',
        cmap='RdYlGn',
        vmin=0,
        vmax=1,
        center=0.5,
        cbar_kws={'label': 'Cosine Similarity'}
    )
    
    plt.title(f'Cross-Head Token Embedding Similarity - Step {step}', fontsize=14, fontweight='bold')
    plt.xlabel('Head B Top Tokens', fontsize=12)
    plt.ylabel('Head A Top Tokens', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


## 6. Extended Analysis with All Prompts


In [ ]:
# Run analysis on all test prompts and collect summary statistics
all_results = []
overlap_stats = []

for prompt_idx, (prompt_a, prompt_b) in enumerate(test_prompts):
    print(f"\n{'#'*80}")
    print(f"Analyzing prompt pair {prompt_idx + 1}/{len(test_prompts)}")
    print(f"A: '{prompt_a}' | B: '{prompt_b}'")
    print(f"{'#'*80}\n")
    
    # Get analysis
    results_pair = get_topk_tokens_analysis(
        model, tokenizer, prompt_a, prompt_b, 
        k=10, max_steps=3, temperature=0.7
    )
    
    all_results.extend(results_pair)
    
    # Compute overlap statistics for this pair
    for step in range(3):
        result_a = [r for r in results_pair if r['step'] == step and r['head'] == 'A'][0]
        result_b = [r for r in results_pair if r['step'] == step and r['head'] == 'B'][0]
        
        tokens_a = set(result_a['topk_tokens'])
        tokens_b = set(result_b['topk_tokens'])
        overlap = tokens_a & tokens_b
        
        # Compute cross-head similarity
        cross_sim = compute_cross_head_similarity(
            result_a['topk_ids'],
            result_b['topk_ids'],
            embedding_layer
        )
        
        overlap_stats.append({
            'prompt_pair': prompt_idx + 1,
            'prompt_a': prompt_a,
            'prompt_b': prompt_b,
            'step': step,
            'overlap_count': len(overlap),
            'overlap_ratio': len(overlap) / 10.0,
            'cross_sim_mean': cross_sim.mean(),
            'cross_sim_max': cross_sim.max(),
            'cross_sim_std': cross_sim.std(),
        })

print(f"\n{'='*80}")
print(f"Collected {len(all_results)} analysis entries")
print(f"{'='*80}")


In [ ]:
# Create summary DataFrame
df_overlap = pd.DataFrame(overlap_stats)

print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)
print("\nOverall statistics across all prompts and steps:")
print(f"  Average token overlap: {df_overlap['overlap_count'].mean():.2f} / 10 ({df_overlap['overlap_ratio'].mean()*100:.1f}%)")
print(f"  Average cross-head similarity: {df_overlap['cross_sim_mean'].mean():.4f}")
print(f"  Max cross-head similarity: {df_overlap['cross_sim_max'].mean():.4f}")

print("\n\nPer-step statistics:")
for step in range(3):
    step_data = df_overlap[df_overlap['step'] == step]
    print(f"\nStep {step}:")
    print(f"  Average overlap: {step_data['overlap_count'].mean():.2f} / 10")
    print(f"  Average cross-sim: {step_data['cross_sim_mean'].mean():.4f}")
    print(f"  Max cross-sim: {step_data['cross_sim_max'].mean():.4f}")

print("\n\nFull overlap statistics table:")
print(df_overlap.to_string(index=False))


In [ ]:
# Visualize statistics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Token overlap by step
ax1 = axes[0, 0]
df_overlap.boxplot(column='overlap_count', by='step', ax=ax1)
ax1.set_title('Token Overlap Distribution by Step', fontsize=12, fontweight='bold')
ax1.set_xlabel('Generation Step')
ax1.set_ylabel('Number of Overlapping Tokens (out of 10)')
ax1.get_figure().suptitle('')  # Remove default title
ax1.grid(True, alpha=0.3)

# Plot 2: Cross-head similarity by step
ax2 = axes[0, 1]
df_overlap.boxplot(column='cross_sim_mean', by='step', ax=ax2)
ax2.set_title('Cross-Head Embedding Similarity by Step', fontsize=12, fontweight='bold')
ax2.set_xlabel('Generation Step')
ax2.set_ylabel('Mean Cosine Similarity')
ax2.get_figure().suptitle('')
ax2.grid(True, alpha=0.3)

# Plot 3: Overlap vs similarity scatter
ax3 = axes[1, 0]
scatter = ax3.scatter(
    df_overlap['overlap_count'],
    df_overlap['cross_sim_mean'],
    c=df_overlap['step'],
    cmap='viridis',
    s=100,
    alpha=0.6
)
ax3.set_xlabel('Token Overlap Count')
ax3.set_ylabel('Mean Cross-Head Similarity')
ax3.set_title('Token Overlap vs Embedding Similarity', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax3)
cbar.set_label('Step')

# Plot 4: Max similarity distribution
ax4 = axes[1, 1]
ax4.hist(df_overlap['cross_sim_max'], bins=15, edgecolor='black', alpha=0.7)
ax4.set_xlabel('Max Cross-Head Similarity')
ax4.set_ylabel('Frequency')
ax4.set_title('Distribution of Max Cross-Head Similarities', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.axvline(df_overlap['cross_sim_max'].mean(), color='red', linestyle='--', 
            label=f'Mean: {df_overlap["cross_sim_max"].mean():.3f}')
ax4.legend()

plt.tight_layout()
plt.show()


## Summary and Conclusions

### Key Findings:

1. **Token Overlap**: The analysis shows how many tokens appear in the top-K for both heads
   - High overlap might indicate the model is predicting "compromise" tokens
   - Low overlap suggests the heads are predicting different tokens for different contexts

2. **Embedding Similarity**: 
   - Within-head similarity shows if top tokens are semantically clustered
   - Cross-head similarity reveals if tokens from different heads are similar in embedding space
   - High cross-head similarity could support the "average embedding" hypothesis

3. **Generation Quality**:
   - If top-K tokens have high embedding similarity across heads, it suggests the model may be predicting semantically ambiguous tokens
   - This could explain why generated text appears incoherent despite low loss

### Next Steps:
- Try different sampling strategies (e.g., increase temperature, use top-p/top-k filtering)
- Investigate if the model benefits from separate embeddings instead of shared ones
- Consider architectural changes to better separate the two streams


In [ ]:
# Optional: Save results to CSV for further analysis
output_path = project_root / 'topk_analysis_results.csv'
df_overlap.to_csv(output_path, index=False)
print(f"Results saved to: {output_path}")
